# Part 2: Agents (lessons 12 to 15)

Groq supports the OpenAI **Responses API** (with function calling), so the course code runs almost unchanged. Differences:
- client: `OpenAI(api_key=GROQ_API_KEY, base_url=groq)` instead of `OpenAI()`
- model: `MODEL = "openai/gpt-oss-120b"` instead of `"gpt-5.4-mini"`
- Groq's `response.output` starts with a `reasoning` item, so we pick the `function_call` item by type instead of `response.output[0]`

## 12 Quick RAG revision

In [1]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "openai/gpt-oss-120b"

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
print(assistant.rag("How do I run Ollama locally?"))

**Running Ollama locally**

1. **Install Ollama**  
   - **macOS** – Download the `.pkg` from https://ollama.com/download and run the installer.  
   - **Windows** – Download the `.msi` from the same page and install it.  
   - **Linux** – Run in a terminal:  

     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model**  
   Open a terminal and execute:

   ```bash
   ollama run llama3
   ```

   This command will:
   - Download the LLaMA 3 model (~4 GB) if it isn’t already present.  
   - Start the model locally.  
   - Open a simple chat‑like prompt where you can type questions.

3. **Verify the local server**  
   In another terminal run:

   ```bash
   curl http://localhost:11434
   ```

   You should receive a JSON response such as:

   ```json
   {"models": [...]}
   ```

   This confirms the Ollama server is listening on port 11434.

4. **(Optional) Use Ollama from Python**  
   Install the client library:

   ```bash
   pip install ollama
   `

Typo: `Olama`. Lexical search finds nothing useful, the fixed pipeline cannot retry.

In [5]:
print([d["question"] for d in assistant.search("How do I run Olama locally?")])
print()
print(assistant.rag("How do I run Olama locally?"))

['Can I run the course locally instead of Codespaces?', 'For peer review, do I need to clone and run the project, or is reading the code enough?', "Why do we need orchestration / Kestra — can't I just run the code in a notebook?", 'How do I use Groq instead of OpenAI in the dlt workshop?', 'How do I know which tables are in the db?']



Yes—you can run the course (including the Olama material) on your own machine instead of using Codespaces.  
All you need to do is set up the same tooling that the course expects:

* Install Python.  
* Install **uv** (the Python package manager used in the repo).  
* Install Jupyter (for the notebook‑based lessons).  
* Install Docker (for any container‑based steps).  
* Install any other utilities that a given module mentions.

Once those pieces are in place you can launch the notebooks, run the scripts, and work through the labs just as you would in Codespaces.  

Be sure to **document your setup** (e.g., record the exact Python version, uv lock file, Docker images, etc.) so that your environment remains reproducible for yourself and anyone else who might need to run it.


## 13 Function calling
### Asking without tools

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model=MODEL,
    input=messages,
)

response.output_text

'Absolutely! You can typically join a course as long as you meet the enrollment requirements and follow the registration process. Here are some general steps you can take:\n\n1. **Check the Course Details**\n   - **Prerequisites:** Make sure you meet any required background knowledge or skills.\n   - **Availability:** Confirm that the course is currently open for enrollment (some courses have limited seats or specific start dates).\n\n2. **Create an Account (if needed)**\n   - Most learning platforms require you to sign up with an email address and set a password.\n\n3. **Enroll in the Course**\n   - Look for an “Enroll,” “Register,” or “Join” button on the course page.\n   - Follow the prompts—some courses are free, while others may require payment or a subscription.\n\n4. **Complete Any Required Forms**\n   - Some courses ask for additional information (e.g., a brief questionnaire, a learning agreement, or a payment method).\n\n5. **Access the Course Materials**\n   - Once enrolled, 

### Defining the tool

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

### Sending the question with the tool

In [9]:
# gpt-oss on Groq sometimes answers directly without searching, and Groq rejects
# tool_choice="required" when the model does not comply. So for this one-step demo we
# add a short developer hint and retry up to 3 times until we get a function_call.
# (Lesson 14 adds full developer instructions; the course model usually searches on its own.)
messages = [{"role": "developer", "content": "Use the search tool to look up course information before answering."}] + messages

for attempt in range(3):
    response = openai_client.responses.create(
        model=MODEL,
        input=messages,
        tools=[search_tool],
    )
    if any(item.type == "function_call" for item in response.output):
        break

[(item.type, getattr(item, "name", None), getattr(item, "arguments", None)) for item in response.output]

[('reasoning', None, None),
 ('function_call',
  'search',
  '{"query":"join course eligibility enrollment prerequisites"}')]

### Executing the function and sending the result back
Course: `call = response.output[0]`. On Groq item 0 is `reasoning`, so select by type.

In [10]:
call = next(item for item in response.output if item.type == "function_call")
args = json.loads(call.arguments)
print(args)

results = search(**args)
result_json = json.dumps(results, indent=2)

{'query': 'join course eligibility enrollment prerequisites'}


In [11]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [12]:
response = openai_client.responses.create(
    model=MODEL,
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

**Yes – you can still join the LLM Zoomcamp!**  

- **Enrollment:** The course is open‑ended, so you can start learning right away even if you missed the original launch. Just head to the course materials (the docs, the GitHub repo, and the course platform) and begin with the first lesson.

- **Certificate:** If you’d like to earn the official completion certificate, you’ll need to **submit a capstone project while the cohort is still accepting submissions**. The certificate isn’t tied to a specific start date, but the project‑submission window does close once the live cohort ends.

- **Next steps:**  
  1. **Read the onboarding docs** – [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/) and the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/).  
  2. **Clone the GitHub repo** – [DataTalksClub/llm-zoomcamp](https://github.com/DataTalksClub/llm-zoomcamp) to get the notebooks and datasets.  
  3. **Create a course‑platform acc

### Token usage and cost
Course numbers ($0.15 / $0.60 per million) happen to equal Groq gpt-oss-120b prices.

In [13]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(871, 418)

In [14]:
def calculate_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_price(usage.input_tokens, usage.output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.00038145


## 14 The agentic loop

In [15]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [16]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

### Processing one response

In [17]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model=MODEL,
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"Can I join the course enrollment eligibility"}


### The full agent loop

In [18]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model=MODEL,
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...


ASSISTANT:
**Answer**

Yes—you can join the LLM Zoomcamp even if you just discovered it now.  
- **Start learning right away** – the course materials are publicly available, so you don’t need to wait for a formal enrollment window.  
- **Certificate eligibility** – if you’d like to earn the course certificate, you’ll need to submit your capstone project (and complete the required peer reviews) before the cohort’s submission deadline. After that date the certificate is no longer issued for new projects.

Feel free to dive into the modules, work on the exercises, and begin planning your final project. If you have any other questions—about the schedule, project requirements, peer‑review process, or anything else—just let me know!


### Wrapping it in a function

In [19]:
def agent_loop(instructions, question, model=MODEL) -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [20]:
_ = agent_loop(instructions, "How do I run Olama locally?")

iteration #1...


function_call: search {"query":"Olama run locally"}
iteration #2...


function_call: search {"query":"Olama"}
iteration #3...


function_call: search {"query":"Olama local installation"}
iteration #4...


function_call: search {"query":"run Ollama locally step by step"}
iteration #5...


ASSISTANT:
**Running Ollama locally – step‑by‑step**

Below is a concise checklist you can follow on any operating system. It covers installation, starting the local model server, verifying that it’s running, and using the Python client (the most common way in the Zoom‑Camp notebooks).

---

## 1️⃣ Install the Ollama runtime

| OS | How to install |
|----|----------------|
| **macOS** | Download the `.pkg` from <https://ollama.com/download> and double‑click to run the installer. |
| **Windows** | Download the `.msi` from the same page and run it. (Make sure you run the installer as an Administrator.) |
| **Linux** | Run the one‑liner in a terminal: <br>`curl -fsSL https://ollama.com/install.sh \| sh` <br> (requires `curl` and `bash`). |

*After the installer finishes, a new `ollama` executable will be on your `PATH`.*

---

## 2️⃣ Pull and run a model locally

The easiest first model is **Llama 3**, but any model listed on the Ollama hub works (e.g., `phi`, `mistral`, `gemma2`).

```ba

In [21]:
_ = agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...


function_call: search {"query":"Can I still join the course after it started"}
iteration #2...


ASSISTANT:
**Yes – you can still join the course!**  
The lectures, notebooks, and all other learning resources are publicly available, so you can start working through the material right away.

If you’d like to earn the **course certificate**, just make sure you submit your **Capstone project** before the submission form closes (the form is the only deadline that matters for certification). Homework assignments are optional, so missing earlier ones won’t prevent you from completing the course or receiving a certificate.

Let me know if you’d like more details about:

- How to access the course materials  
- Project submission guidelines and deadlines  
- Getting help with the homework or capstone  
- Anything else about the Zoomcamp (schedule, community forums, etc.)

I’m happy to help you get started!


### Encouraging multiple searches

In [22]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

_ = agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...


function_call: search {"query":"join course enrollment eligibility"}
iteration #2...


ASSISTANT:
**Short answer:** Yes—you can join the LLM Zoomcamp even if you just discovered it now.

**What you need to know**

| Topic | Details |
|-------|---------|
| **Can I enroll now?** | Absolutely. The course is open‑ended; you can start watching the videos and working through the notebooks immediately. |
| **Do I need to register first?** | Registration is optional and only used to gauge interest. You can begin learning without filling out the form. |
| **Will I get a certificate?** | A certificate is only issued if you submit a capstone project (and any required peer reviews) while the cohort is still accepting submissions. So you can join now, but to earn the certificate you must complete and turn in the project before the submission deadline. |
| **Self‑paced vs. live cohort** | You may work through the material at your own pace, but the official certificate requires you to finish the project during a live cohort period. Homework is optional for certification. |
| **Where to

### Restricting off-topic questions

In [23]:
_ = agent_loop(instructions, "what's queen gambit?")

iteration #1...


function_call: search {"query":"Queen's Gambit chess opening"}
iteration #2...


function_call: search {"query":"What is the Queen's Gambit?"}
iteration #3...


ASSISTANT:
**The Queen’s Gambit – a classic chess opening**

---

### 1. What it is
The **Queen’s Gambit** is one of the oldest and most respected openings in chess. It begins with the moves:

1. **d4 d5**  
2. **c4**

White offers a pawn on **c4** (the “gambit”) to tempt Black into giving up the center. The idea is not so much to win the pawn outright as to gain rapid development, control of the centre, and open lines for the pieces.

---

### 2. Main ideas for each side  

| White (who plays the gambit) | Black (who can accept or decline) |
|------------------------------|-----------------------------------|
| • Take the initiative in the centre with **e4** or **c4‑d5** exchanges.  <br>• Open the c‑file for the queen’s rook. <br>• Develop the knights to **c3** and **f3**, bishop to **g2** (or **e2**), and castle early. | • Decide whether to **accept** the pawn (**dxc4**) or **decline** it (e.g., **e6**, **c6**, **e5**). <br>• If accepting, try to hold onto the extra pawn with moves l

In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

_ = agent_loop(instructions, "what's queen gambit?")

iteration #1...


function_call: search {"query":"queen gambit"}
iteration #2...


ASSISTANT:
I’m sorry, but that question isn’t related to the course material or its logistics, so I don’t have an answer for it.  

Is there something else about the course that you’d like help with?


## 15 ToyAIKit
Course: `OpenAIClient(model="gpt-5.4-mini")`. We pass `client=openai_client` (Groq) and `model=MODEL`.

In [25]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [26]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [27]:
from toyaikit.pricing import PricingConfig

# toyaikit has no price for the Groq model, so result.cost would be None.
# Register it (USD per 1M tokens, from console.groq.com/docs/model/openai/gpt-oss-120b).
pricing = PricingConfig()
pricing.register_model(MODEL, input_price=0.15, output_price=0.60)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model=MODEL, client=openai_client),
    pricing_config=pricing,
)

In [28]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [29]:
result.cost

CostInfo(input_cost=Decimal('0.00066615'), output_cost=Decimal('0.0004134'), total_cost=Decimal('0.00107955'))

In [30]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01m5y1sw53e2asx3gjx3h3w3ft', summary=[], type='reasoning', content=[Content(t

In [31]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


Step,Command,What it does
List installed models,ollama list,Shows models you already have
Browse available models,Open https://ollama.com/library,Pick a new model name
Pull a new model,ollama pull <model>,Downloads the model files
Run a model interactively,ollama run <model>,Starts a local chat session
Use from Python,"ollama.chat(model=""<model>"", …)",Calls the model via the client


### Interactive chat
Run this yourself in VS Code (it waits for typed input; type `stop` to exit). Left commented so batch runs do not hang.

In [32]:
# runner.run()